## 0. opendataloader-pdf ✅ 推荐（纯 CPU，快）

`pip install opendataloader-pdf` — 基于规则的结构化解析，不用 GPU、不跑大模型。
本机实测 4 页约 3.4 秒，文本/表格/图片全部提取。

> 底层是 Java CLI，需要 Java 运行时。本机已用 conda 装了 OpenJDK（`/home/mvr/miniconda3/bin`），
> 下面代码会把它加到 PATH，所以 notebook kernel 能找到 `java`。

In [8]:
# opendataloader-pdf：纯 CPU、快
# pip install opendataloader-pdf
import os

# java 依赖
# (bash)$ sudo apt install openjdk-17-jdk

import opendataloader_pdf

PDF_PATH   = "/mnt/d/ddesktop/FileParser/sources/AGENTSmd_may_nouse.pdf"
OUTPUT_DIR = "/mnt/d/ddesktop/FileParser/output_opendataloader"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 输出 <文件名>.md / .json，图片抽到 <文件名>_images/ 目录
opendataloader_pdf.convert(
    input_path=PDF_PATH,
    output_dir=OUTPUT_DIR,
    format=["markdown", "json"],
)

print("完成 →", OUTPUT_DIR)

Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor preprocessing
INFO: File name: /mnt/d/ddesktop/FileParser/sources/AGENTSmd_may_nouse.pdf
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Number of pages: 4
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Author: null
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Title: (1) X
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Creation date: D:20260608065110+00'00'
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor calculateDocumentInfo
INFO: Modification date: D:20260608065110+00'00'
Jun 25, 2026 4:54:20 PM org.opendataloader.pdf.processors.DocumentProcessor processDocument
INFO: Processing 4 pages with 1 threads
Jun 25, 2026 4:54:22 PM org.opendataloader.pdf.json.Json

## 1. MinerU CLI ⚠️ 需要 GPU
`pip install "mineru[all]"` — CLI 解析 PDF → Markdown

> MinerU2.5-Pro 是 1.2B 视觉大模型（VLM）。本机无 GPU，CPU 上实测 **每页约 16 分钟**，基本不可用。
> 有 GPU 时再用；CPU 场景请用上面的 opendataloader-pdf。

In [6]:
# examples: mineru  --pypi
# pip install "mineru[all]"
#
# 本地模型配置：通过 `~/mineru.json` 指定，不是通过环境变量
# {
#     "model-source": "local",
#     "models-dir": { "vlm": "/mnt/e/local_models/ocr/OpenDataLab--MinerU2.5-Pro-2605-1.2B" }
# }
#
# -b / --backend: pipeline | vlm-auto-engine | hybrid-auto-engine | vlm-http-client | hybrid-http-client
# 本地 VLM 模型（MinerU2.5-Pro）对应 -b vlm-auto-engine

import os
import subprocess

PDF_PATH   = "/mnt/d/ddesktop/FileParser/sources/AGENTSmd_may_nouse.pdf"
OUTPUT_DIR = "/mnt/d/ddesktop/FileParser/output_mineru"

os.makedirs(OUTPUT_DIR, exist_ok=True)

result = subprocess.run(
    ["mineru", "-p", PDF_PATH, "-o", OUTPUT_DIR, "-b", "vlm-auto-engine"],
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

Start MinerU FastAPI Service: http://127.0.0.1:43113
API documentation: http://127.0.0.1:43113/docs

STDERR: 2026-06-25 16:02:58.539 | INFO     | mineru.cli.client:run_orchestrated_cli:953 - Started local mineru-api at http://127.0.0.1:43113
2026-06-25 16:03:00.802 | INFO     | __main__:create_app:236 - Request concurrency limited to 3
INFO:     Started server process [11416]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:43113 (Press CTRL+C to quit)
2026-06-25 16:03:01.326 | INFO     | mineru.cli.client:run_planned_task:832 - Submitting batch 1/1 | 1 document, 4 pages in this batch | 4 pages total | task#1 [AGENTSmd_may_nouse]
2026-06-25 16:03:01.388 | INFO     | mineru.utils.engine_utils:get_vlm_engine:34 - Using transformers as the inference engine for VLM.

Loading weights: 100%|██████████| 681/681 [00:00<00:00, 1583.15it/s]
[transformers] Qwen2VLForConditionalGeneration LOAD REPORT from: /mnt/e/local

In [ ]:
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image
from mineru_vl_utils import MinerUClient

mineru_path = "/mnt/e/local_models/ocr/OpenDataLab--MinerU2.5-Pro-2605-1.2B"

# for transformers>=4.56.0
model = Qwen2VLForConditionalGeneration.from_pretrained(
    mineru_path, dtype="auto", device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    mineru_path, use_fast=True
)

client = MinerUClient(
    backend="transformers", model=model, processor=processor,
    image_analysis=False # default False, set True to enable image/chart analysis
)

# print(client.two_step_extract(Image.open("/path/to/page.png")))


## 2. PaddleOCR-VL-1.6 (transformers) ⚠️ 需要 GPU
`pip install "transformers>=5.0.0" torch pillow pymupdf` — 逐页 OCR / 表格 / 公式识别

> 同样是视觉大模型，CPU 上很慢。有 GPU 时再用。

In [ ]:
# examples: paddleocr: ppocr-vl 1.6
# model_path: /mnt/e/local_models/ocr/PaddlePaddle--PaddleOCR-VL-1.6
# pip install "transformers>=5.0.0" torch pillow pymupdf

import os
import fitz
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

PDF_PATH       = "/mnt/d/ddesktop/FileParser/sources/AGENTSmd_may_nouse.pdf"
MODEL_PATH     = "/mnt/e/local_models/ocr/PaddlePaddle--PaddleOCR-VL-1.6"
OUTPUT_DIR     = "/mnt/d/ddesktop/FileParser/output_paddleocr"
OUTPUT_MD_FILE = os.path.join(OUTPUT_DIR, "paddleocr_vl_output.md")

# 支持的任务: 'ocr' | 'table' | 'formula' | 'chart' | 'spotting' | 'seal'
TASK = "ocr"

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 加载本地模型
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"使用设备: {DEVICE}")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, torch_dtype=torch.bfloat16
).to(DEVICE).eval()
processor = AutoProcessor.from_pretrained(MODEL_PATH)

In [ ]:
PROMPTS = {
    "ocr":      "OCR:",
    "table":    "Table Recognition:",
    "formula":  "Formula Recognition:",
    "chart":    "Chart Recognition:",
    "spotting": "Spotting:",
    "seal":     "Seal Recognition:",
}

def parse_page(image: Image.Image, task: str = "ocr") -> str:
    max_pixels = 2048 * 28 * 28 if task == "spotting" else 1280 * 28 * 28
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": PROMPTS[task]},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        images_kwargs={
            "size": {
                "shortest_edge": processor.image_processor.min_pixels,
                "longest_edge":  max_pixels,
            }
        },
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=2048)

    return processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])

In [ ]:
# PDF → 页面图像 → 逐页 OCR → 合并保存
doc = fitz.open(PDF_PATH)
results = []

for i, page in enumerate(doc):
    pix = page.get_pixmap(dpi=150)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    text = parse_page(img, task=TASK)
    results.append(text)
    print(f"[{i+1}/{len(doc)}] 完成")

doc.close()

with open(OUTPUT_MD_FILE, "w", encoding="utf-8") as f:
    f.write("\n\n---\n\n".join(results))

print(f"已保存: {OUTPUT_MD_FILE}")